# Feature Scaling and Normalization in Machine Learning

This notebook provides a comprehensive overview of feature scaling and normalization techniques used in machine learning, including their implementation, visualization, and best practices.

**Topics Covered:**
- Understanding why feature scaling matters
- Min-Max Scaling (Normalization)
- Standardization (Z-score Normalization)
- Robust Scaling
- Unit Vector Normalization
- Log Transformation
- Comparing different scaling methods
- Integrating scaling in ML pipelines
- Handling outliers
- Guidelines for choosing the right scaling method

## 1. Import Required Libraries

We'll start by importing the essential Python libraries needed for this notebook.

In [ ]:
# Core libraries for data manipulation and analysis
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

# Machine learning and preprocessing
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, Normalizer
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs, make_classification, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

# Set the style for our plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook")

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## 2. Understand the Need for Feature Scaling

Feature scaling is a crucial preprocessing step in machine learning. Many algorithms are sensitive to the scale of the input features, and having features with different scales can negatively impact model performance.

### Why Feature Scaling Matters:

1. **Gradient Descent-Based Algorithms**: Algorithms that use gradient descent (like linear regression, logistic regression, neural networks) converge much faster with scaled features.

2. **Distance-Based Algorithms**: Models like k-NN, k-means, and SVM use distance calculations, which can be dominated by features with larger scales if not normalized.

3. **Regularization**: L1 and L2 regularization are affected by feature scales, as penalties are applied based on coefficient magnitudes.

Let's demonstrate this with a visual example.

In [ ]:
# Create a dataset with features of different scales
np.random.seed(42)
n_samples = 1000

# Feature 1: Small scale (values between 0 and 1)
X1 = np.random.random(n_samples)

# Feature 2: Large scale (values between 0 and 1000)
X2 = np.random.random(n_samples) * 1000

# Combine features into a single array
X = np.column_stack((X1, X2))

# Create a scatter plot to visualize the difference in scales
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], alpha=0.5)
plt.title('Original Data (Different Scales)')
plt.xlabel('Feature 1 (0-1 scale)')
plt.ylabel('Feature 2 (0-1000 scale)')

plt.subplot(1, 2, 2)
# Scale the data using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], alpha=0.5)
plt.title('Scaled Data (Same Scale)')
plt.xlabel('Feature 1 (standardized)')
plt.ylabel('Feature 2 (standardized)')

plt.tight_layout()
plt.show()

### Impact on ML Algorithms

Let's see how the difference in feature scales affects the performance of a distance-based algorithm like K-Nearest Neighbors.

In [ ]:
# Create a classification dataset
X, y = make_blobs(n_samples=1000, centers=3, n_features=2, random_state=42)

# Make one feature have a much larger scale
X[:, 1] = X[:, 1] * 100

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train a KNN classifier on the original (unscaled) data
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_train, y_train)
y_pred_unscaled = knn_unscaled.predict(X_test)
acc_unscaled = accuracy_score(y_test, y_pred_unscaled)

# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train a KNN classifier on the scaled data
knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = knn_scaled.predict(X_test_scaled)
acc_scaled = accuracy_score(y_test, y_pred_scaled)

# Visualize the decision boundaries
def plot_decision_boundary(X, y, model, title):
    h = 0.02  # Step size in the mesh
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF']))
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=ListedColormap(['#FF0000', '#00FF00', '#0000FF']), edgecolors='k')
    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')

# Create plots
plt.figure(figsize=(14, 6))

# Original unscaled data - this may look distorted due to scale differences
plt.subplot(1, 2, 1)
plot_decision_boundary(X_test, y_test, knn_unscaled, f'Unscaled Data\nAccuracy: {acc_unscaled:.3f}')

# Scaled data plot
plt.subplot(1, 2, 2)
plot_decision_boundary(X_test_scaled, y_test, knn_scaled, f'Scaled Data\nAccuracy: {acc_scaled:.3f}')

plt.tight_layout()
plt.show()

## 3. Generate Example Dataset

To understand and compare different scaling methods, let's create a synthetic dataset with features having different scales and distributions. We'll use this dataset throughout the notebook.

In [ ]:
# Generate a synthetic dataset
def create_synthetic_data(n_samples=1000, with_outliers=False):
    np.random.seed(42)
    
    # Feature 1: Normal distribution (μ=5, σ=2)
    f1 = np.random.normal(loc=5, scale=2, size=n_samples)
    
    # Feature 2: Large scale feature (μ=100, σ=20)
    f2 = np.random.normal(loc=100, scale=20, size=n_samples)
    
    # Feature 3: Small scale feature (μ=0.1, σ=0.01)
    f3 = np.random.normal(loc=0.1, scale=0.01, size=n_samples)
    
    # Feature 4: Exponentially distributed feature
    f4 = np.random.exponential(scale=5, size=n_samples)
    
    # Feature 5: Uniform distribution
    f5 = np.random.uniform(low=0, high=10, size=n_samples)
    
    # Add outliers if requested
    if with_outliers:
        # Add outliers to 1% of the data
        outlier_indices = np.random.choice(n_samples, size=int(0.01 * n_samples), replace=False)
        f1[outlier_indices] = f1[outlier_indices] * 5
        f2[outlier_indices] = f2[outlier_indices] * 5
        f3[outlier_indices] = f3[outlier_indices] * 5
        f4[outlier_indices] = f4[outlier_indices] * 10
        f5[outlier_indices] = f5[outlier_indices] * 3
    
    # Combine all features into a DataFrame
    df = pd.DataFrame({
        'normal_med': f1,
        'normal_large': f2,
        'normal_small': f3,
        'exponential': f4,
        'uniform': f5
    })
    
    return df

# Generate two datasets: one without outliers and one with outliers
df_clean = create_synthetic_data(with_outliers=False)
df_with_outliers = create_synthetic_data(with_outliers=True)

# Display summary statistics
print("Dataset without outliers:")
print(df_clean.describe())
print("\nDataset with outliers:")
print(df_with_outliers.describe())

# Visualize the distributions of features
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

features = df_clean.columns

# Plot clean data
for i, feature in enumerate(features):
    sns.histplot(df_clean[feature], kde=True, ax=axes[0, i])
    axes[0, i].set_title(f'Clean - {feature}')
    axes[0, i].set_xlabel('')

# Plot data with outliers
for i, feature in enumerate(features):
    sns.histplot(df_with_outliers[feature], kde=True, ax=axes[1, i])
    axes[1, i].set_title(f'With outliers - {feature}')

plt.tight_layout()
plt.show()

# Box plots to better visualize the outliers
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=df_clean, ax=axes[0])
axes[0].set_title('Data without Outliers')
axes[0].set_yscale('symlog')  # Using symlog scale to handle different scales

sns.boxplot(data=df_with_outliers, ax=axes[1])
axes[1].set_title('Data with Outliers')
axes[1].set_yscale('symlog')  # Using symlog scale to handle different scales

plt.tight_layout()
plt.show()

## 4. Min-Max Scaling

Min-Max scaling (often called normalization) is a method that scales features to a fixed range, usually [0, 1] or [-1, 1]. The transformation is given by:

$X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}$

For scaling to range [a, b], we use:

$X_{scaled} = a + \frac{(X - X_{min}) \times (b - a)}{X_{max} - X_{min}}$

**Advantages:**
- Preserves relationships between the original data values exactly
- Bounds values to a specific range
- Works well when the distribution is not Gaussian or the standard deviation is very small

**Limitations:**
- Highly sensitive to outliers

In [ ]:
# Manual implementation of Min-Max scaling
def min_max_scale_manual(data, feature_range=(0, 1)):
    a, b = feature_range
    data_min = data.min()
    data_max = data.max()
    
    # Handle potential division by zero
    denominator = (data_max - data_min)
    denominator = np.where(denominator != 0, denominator, 1)
    
    scaled = a + (data - data_min) * (b - a) / denominator
    return scaled

# Apply manual Min-Max scaling to clean data
df_clean_minmax_manual = pd.DataFrame()
for column in df_clean.columns:
    df_clean_minmax_manual[column] = min_max_scale_manual(df_clean[column])

# Using scikit-learn's MinMaxScaler
minmax_scaler = MinMaxScaler(feature_range=(0, 1))
df_clean_minmax_sklearn = pd.DataFrame(
    minmax_scaler.fit_transform(df_clean),
    columns=df_clean.columns
)

# Apply to data with outliers
minmax_scaler_outliers = MinMaxScaler(feature_range=(0, 1))
df_outliers_minmax_sklearn = pd.DataFrame(
    minmax_scaler_outliers.fit_transform(df_with_outliers),
    columns=df_with_outliers.columns
)

# Visualize the results
fig, axes = plt.subplots(3, 5, figsize=(20, 12))

# Original clean data
for i, feature in enumerate(df_clean.columns):
    sns.histplot(df_clean[feature], kde=True, ax=axes[0, i])
    axes[0, i].set_title(f'Original - {feature}')

# Min-Max scaled clean data
for i, feature in enumerate(df_clean_minmax_sklearn.columns):
    sns.histplot(df_clean_minmax_sklearn[feature], kde=True, ax=axes[1, i])
    axes[1, i].set_title(f'Min-Max Scaled (Clean) - {feature}')

# Min-Max scaled data with outliers
for i, feature in enumerate(df_outliers_minmax_sklearn.columns):
    sns.histplot(df_outliers_minmax_sklearn[feature], kde=True, ax=axes[2, i])
    axes[2, i].set_title(f'Min-Max Scaled (With Outliers) - {feature}')

plt.tight_layout()
plt.show()

# Compare the distributions using box plots
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Original data
sns.boxplot(data=df_clean, ax=axes[0])
axes[0].set_title('Original Data (Clean)')

# Min-Max scaled clean data
sns.boxplot(data=df_clean_minmax_sklearn, ax=axes[1])
axes[1].set_title('Min-Max Scaled Data (Clean)')

# Min-Max scaled data with outliers
sns.boxplot(data=df_outliers_minmax_sklearn, ax=axes[2])
axes[2].set_title('Min-Max Scaled Data (With Outliers)')

plt.tight_layout()
plt.show()

### Effect of Outliers on Min-Max Scaling

Notice how outliers significantly affect Min-Max scaling. When outliers are present, most of the data gets compressed into a small range, reducing the resolution for the majority of the data points. This is one of the main limitations of Min-Max scaling.

## 5. Standardization (Z-score Normalization)

Standardization (or Z-score normalization) transforms features to have zero mean and unit variance. The formula is:

$X_{scaled} = \frac{X - \mu}{\sigma}$

Where:
- $\mu$ is the mean of the feature values
- $\sigma$ is the standard deviation

**Advantages:**
- Less affected by outliers than Min-Max scaling
- Useful for algorithms that assume features are normally distributed
- Helpful for training algorithms that use gradient descent

**Limitations:**
- Doesn't produce normalized values within a specific range
- Still influenced by outliers (though less than Min-Max scaling)

In [ ]:
# Manual implementation of Standardization
def standardize_manual(data):
    mean = np.mean(data)
    std = np.std(data)
    
    # Handle potential division by zero
    std = std if std != 0 else 1
    
    return (data - mean) / std

# Apply manual standardization to clean data
df_clean_standard_manual = pd.DataFrame()
for column in df_clean.columns:
    df_clean_standard_manual[column] = standardize_manual(df_clean[column])

# Using scikit-learn's StandardScaler
std_scaler = StandardScaler()
df_clean_standard_sklearn = pd.DataFrame(
    std_scaler.fit_transform(df_clean),
    columns=df_clean.columns
)

# Apply to data with outliers
std_scaler_outliers = StandardScaler()
df_outliers_standard_sklearn = pd.DataFrame(
    std_scaler_outliers.fit_transform(df_with_outliers),
    columns=df_with_outliers.columns
)

# Visualize the results
fig, axes = plt.subplots(3, 5, figsize=(20, 12))

# Original clean data
for i, feature in enumerate(df_clean.columns):
    sns.histplot(df_clean[feature], kde=True, ax=axes[0, i])
    axes[0, i].set_title(f'Original - {feature}')

# Standardized clean data
for i, feature in enumerate(df_clean_standard_sklearn.columns):
    sns.histplot(df_clean_standard_sklearn[feature], kde=True, ax=axes[1, i])
    axes[1, i].set_title(f'Standardized (Clean) - {feature}')

# Standardized data with outliers
for i, feature in enumerate(df_outliers_standard_sklearn.columns):
    sns.histplot(df_outliers_standard_sklearn[feature], kde=True, ax=axes[2, i])
    axes[2, i].set_title(f'Standardized (With Outliers) - {feature}')

plt.tight_layout()
plt.show()

# Compare the distributions using box plots
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Original data
sns.boxplot(data=df_clean, ax=axes[0])
axes[0].set_title('Original Data (Clean)')

# Standardized clean data
sns.boxplot(data=df_clean_standard_sklearn, ax=axes[1])
axes[1].set_title('Standardized Data (Clean)')

# Standardized data with outliers
sns.boxplot(data=df_outliers_standard_sklearn, ax=axes[2])
axes[2].set_title('Standardized Data (With Outliers)')

plt.tight_layout()
plt.show()

## 6. Robust Scaling

Robust scaling is a technique that's less sensitive to outliers. It uses the median and interquartile range (IQR) instead of the mean and standard deviation:

$X_{scaled} = \frac{X - median(X)}{IQR(X)}$

Where:
- $median(X)$ is the median of the feature values
- $IQR(X)$ is the interquartile range (75th percentile - 25th percentile)

**Advantages:**
- Highly robust to outliers
- Works well when there are outliers that shouldn't be removed
- Preserves the shape of the distribution better than standardization for non-normal data

**Limitations:**
- May not be as effective for normally distributed data without outliers

In [ ]:
# Manual implementation of Robust Scaling
def robust_scale_manual(data):
    median = np.median(data)
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)
    iqr = q3 - q1
    
    # Handle potential division by zero
    iqr = iqr if iqr != 0 else 1
    
    return (data - median) / iqr

# Apply manual robust scaling to clean data
df_clean_robust_manual = pd.DataFrame()
for column in df_clean.columns:
    df_clean_robust_manual[column] = robust_scale_manual(df_clean[column])

# Using scikit-learn's RobustScaler
robust_scaler = RobustScaler()
df_clean_robust_sklearn = pd.DataFrame(
    robust_scaler.fit_transform(df_clean),
    columns=df_clean.columns
)

# Apply to data with outliers
robust_scaler_outliers = RobustScaler()
df_outliers_robust_sklearn = pd.DataFrame(
    robust_scaler_outliers.fit_transform(df_with_outliers),
    columns=df_with_outliers.columns
)

# Visualize the results
fig, axes = plt.subplots(3, 5, figsize=(20, 12))

# Original clean data
for i, feature in enumerate(df_clean.columns):
    sns.histplot(df_clean[feature], kde=True, ax=axes[0, i])
    axes[0, i].set_title(f'Original - {feature}')

# Robust scaled clean data
for i, feature in enumerate(df_clean_robust_sklearn.columns):
    sns.histplot(df_clean_robust_sklearn[feature], kde=True, ax=axes[1, i])
    axes[1, i].set_title(f'Robust Scaled (Clean) - {feature}')

# Robust scaled data with outliers
for i, feature in enumerate(df_outliers_robust_sklearn.columns):
    sns.histplot(df_outliers_robust_sklearn[feature], kde=True, ax=axes[2, i])
    axes[2, i].set_title(f'Robust Scaled (With Outliers) - {feature}')

plt.tight_layout()
plt.show()

# Compare the distributions using box plots
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Original data with outliers
sns.boxplot(data=df_with_outliers, ax=axes[0])
axes[0].set_title('Original Data (With Outliers)')

# StandardScaler applied to data with outliers
sns.boxplot(data=df_outliers_standard_sklearn, ax=axes[1])
axes[1].set_title('StandardScaler (With Outliers)')

# RobustScaler applied to data with outliers
sns.boxplot(data=df_outliers_robust_sklearn, ax=axes[2])
axes[2].set_title('RobustScaler (With Outliers)')

plt.tight_layout()
plt.show()

## 7. Unit Vector Normalization

Unit vector normalization scales individual samples to have unit norm (length 1). This is particularly useful for algorithms that depend on the dot product between samples, such as cosine similarity in text analysis.

The most common norms used are:
- L1 norm: Sum of absolute values
- L2 norm: Euclidean distance (square root of sum of squared values)
- L-infinity (max) norm: Maximum absolute value

The formula for L2 normalization is:
$X_{scaled} = \frac{X}{||X||_2}$ where $||X||_2 = \sqrt{\sum_{i=1}^{n} X_i^2}$

**Advantages:**
- Useful for text data and comparing documents
- Helpful for SVM algorithms and cosine similarity calculations
- Focuses on the direction of the data, not the magnitude

**Limitations:**
- Doesn't address the scale differences between features
- Only considers the relationship between features within a sample

In [ ]:
# Manual implementation of L2 normalization
def normalize_l2_manual(data):
    # Calculate L2 norm (Euclidean norm)
    l2_norm = np.sqrt(np.sum(data ** 2))
    
    # Handle potential division by zero
    l2_norm = l2_norm if l2_norm != 0 else 1
    
    return data / l2_norm

# Create example data for normalization demonstration
# We'll use a small subset of the clean data
sample_data = df_clean.iloc[:5].copy()
print("Original sample data:")
print(sample_data)

# Manually normalize each sample (row) using L2 norm
normalized_manual = np.zeros_like(sample_data.values)
for i in range(len(sample_data)):
    normalized_manual[i] = normalize_l2_manual(sample_data.iloc[i].values)

df_normalized_manual = pd.DataFrame(normalized_manual, columns=sample_data.columns)
print("\nManually normalized data (L2 norm):")
print(df_normalized_manual)

# Calculate and print the L2 norm of each row to verify it equals 1
l2_norms_manual = np.sqrt(np.sum(df_normalized_manual.values ** 2, axis=1))
print("\nL2 norms after manual normalization:")
print(l2_norms_manual)

# Using scikit-learn's Normalizer (L2 norm by default)
normalizer = Normalizer()
df_normalized_sklearn = pd.DataFrame(
    normalizer.fit_transform(sample_data),
    columns=sample_data.columns
)

print("\nNormalized data using sklearn (L2 norm):")
print(df_normalized_sklearn)

# Calculate and print the L2 norm of each row to verify it equals 1
l2_norms_sklearn = np.sqrt(np.sum(df_normalized_sklearn.values ** 2, axis=1))
print("\nL2 norms after sklearn normalization:")
print(l2_norms_sklearn)

# Apply normalization to the full datasets
normalizer_clean = Normalizer()
df_clean_normalized = pd.DataFrame(
    normalizer_clean.fit_transform(df_clean),
    columns=df_clean.columns
)

normalizer_outliers = Normalizer()
df_outliers_normalized = pd.DataFrame(
    normalizer_outliers.fit_transform(df_with_outliers),
    columns=df_with_outliers.columns
)

# Visualize the results with histograms
fig, axes = plt.subplots(3, 5, figsize=(20, 12))

# Original clean data
for i, feature in enumerate(df_clean.columns):
    sns.histplot(df_clean[feature], kde=True, ax=axes[0, i])
    axes[0, i].set_title(f'Original - {feature}')

# Normalized clean data
for i, feature in enumerate(df_clean_normalized.columns):
    sns.histplot(df_clean_normalized[feature], kde=True, ax=axes[1, i])
    axes[1, i].set_title(f'Normalized (Clean) - {feature}')

# Normalized data with outliers
for i, feature in enumerate(df_outliers_normalized.columns):
    sns.histplot(df_outliers_normalized[feature], kde=True, ax=axes[2, i])
    axes[2, i].set_title(f'Normalized (With Outliers) - {feature}')

plt.tight_layout()
plt.show()

# Compare with PCA to see the direction preservation
# For this we'll use a 2D projection to visualize
pca = PCA(n_components=2)
X_original_2d = pca.fit_transform(df_clean)
X_normalized_2d = pca.fit_transform(df_clean_normalized)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_original_2d[:, 0], X_original_2d[:, 1], alpha=0.6)
plt.title('Original Data - PCA Projection')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

plt.subplot(1, 2, 2)
plt.scatter(X_normalized_2d[:, 0], X_normalized_2d[:, 1], alpha=0.6)
plt.title('Normalized Data - PCA Projection')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

plt.tight_layout()
plt.show()

## 8. Log Transformation

Log transformation is useful for handling highly skewed data and can help make patterns in the data more interpretable. It's often used with right-skewed data or data with exponential growth patterns.

Common log transformations include:
- Natural log (ln)
- Log base 10 (log10)
- Log base 2 (log2)

**Advantages:**
- Reduces the effect of outliers
- Makes highly skewed distributions more symmetric
- Often makes non-linear relationships more linear
- Helps with multiplicative relationships in data

**Limitations:**
- Cannot be applied to zero or negative values directly (need to add a small constant)
- Transforms the data in a non-linear way, which may affect interpretability

In [ ]:
# Let's focus on the exponential feature which has a right-skewed distribution
exp_feature = df_clean['exponential'].copy()
exp_feature_outliers = df_with_outliers['exponential'].copy()

# Applying different log transformations
# We need to be careful with zeros or negative values
# Natural log (ln)
log_natural = np.log(exp_feature)
log_natural_outliers = np.log(exp_feature_outliers)

# Log base 10
log10 = np.log10(exp_feature)
log10_outliers = np.log10(exp_feature_outliers)

# Log base 2
log2 = np.log2(exp_feature)
log2_outliers = np.log2(exp_feature_outliers)

# Visualize the transformations
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Original data without outliers
sns.histplot(exp_feature, kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Original (Clean)')

# Log transformations for clean data
sns.histplot(log_natural, kde=True, ax=axes[0, 1])
axes[0, 1].set_title('Natural Log (ln) - Clean')

sns.histplot(log10, kde=True, ax=axes[0, 2])
axes[0, 2].set_title('Log Base 10 - Clean')

sns.histplot(log2, kde=True, ax=axes[0, 3])
axes[0, 3].set_title('Log Base 2 - Clean')

# Original data with outliers
sns.histplot(exp_feature_outliers, kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Original (With Outliers)')

# Log transformations for data with outliers
sns.histplot(log_natural_outliers, kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Natural Log (ln) - With Outliers')

sns.histplot(log10_outliers, kde=True, ax=axes[1, 2])
axes[1, 2].set_title('Log Base 10 - With Outliers')

sns.histplot(log2_outliers, kde=True, ax=axes[1, 3])
axes[1, 3].set_title('Log Base 2 - With Outliers')

plt.tight_layout()
plt.show()

# Create a more complex example with a feature having exponential relationship
np.random.seed(42)
x = np.random.uniform(0, 10, 1000)
y_exp = np.exp(x/3) + np.random.normal(0, 20, 1000)  # Exponential relationship with noise

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(x, y_exp, alpha=0.6)
plt.title('Original Data (Exponential Relationship)')
plt.xlabel('x')
plt.ylabel('y')

plt.subplot(1, 2, 2)
plt.scatter(x, np.log(y_exp), alpha=0.6)
plt.title('After Log Transformation of y')
plt.xlabel('x')
plt.ylabel('log(y)')

plt.tight_layout()
plt.show()

## 9. Comparing Scaling Methods

Let's compare all the scaling methods we've covered so far to understand their effects on data distribution, outlier sensitivity, and relationship preservation.

We'll visualize:
1. How each method transforms the data distribution
2. How outliers affect each scaling method
3. How each method preserves the relationships between features

In [ ]:
# Let's use a small subset of the data for easier comparison
# We'll choose two features with different scales
comparison_df_clean = df_clean[['normal_med', 'normal_large']].iloc[:100].copy()
comparison_df_outliers = df_with_outliers[['normal_med', 'normal_large']].iloc[:100].copy()

# Add an outlier to make the visualization clearer
comparison_df_outliers.iloc[0, 0] = 20  # Add outlier to normal_med
comparison_df_outliers.iloc[0, 1] = 300  # Add outlier to normal_large

# Apply different scaling methods
# 1. Min-Max Scaling
minmax_scaler = MinMaxScaler()
comparison_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(comparison_df_outliers),
    columns=comparison_df_outliers.columns
)

# 2. Standardization
std_scaler = StandardScaler()
comparison_std = pd.DataFrame(
    std_scaler.fit_transform(comparison_df_outliers),
    columns=comparison_df_outliers.columns
)

# 3. Robust Scaling
robust_scaler = RobustScaler()
comparison_robust = pd.DataFrame(
    robust_scaler.fit_transform(comparison_df_outliers),
    columns=comparison_df_outliers.columns
)

# 4. Log Transformation (add a small constant to avoid log(0))
comparison_log = comparison_df_outliers.copy()
comparison_log = np.log1p(comparison_log)  # log(1+x) to handle zeros

# 5. Normalization (L2)
normalizer = Normalizer()
comparison_norm = pd.DataFrame(
    normalizer.fit_transform(comparison_df_outliers),
    columns=comparison_df_outliers.columns
)

# Create scatter plots to see the relationship between the two features
plt.figure(figsize=(18, 15))

# Original data
plt.subplot(3, 2, 1)
plt.scatter(comparison_df_outliers['normal_med'], comparison_df_outliers['normal_large'])
plt.title('Original Data')
plt.xlabel('normal_med')
plt.ylabel('normal_large')

# Min-Max Scaled
plt.subplot(3, 2, 2)
plt.scatter(comparison_minmax['normal_med'], comparison_minmax['normal_large'])
plt.title('Min-Max Scaling')
plt.xlabel('normal_med (scaled)')
plt.ylabel('normal_large (scaled)')

# Standardized
plt.subplot(3, 2, 3)
plt.scatter(comparison_std['normal_med'], comparison_std['normal_large'])
plt.title('Standardization')
plt.xlabel('normal_med (scaled)')
plt.ylabel('normal_large (scaled)')

# Robust Scaled
plt.subplot(3, 2, 4)
plt.scatter(comparison_robust['normal_med'], comparison_robust['normal_large'])
plt.title('Robust Scaling')
plt.xlabel('normal_med (scaled)')
plt.ylabel('normal_large (scaled)')

# Log Transformed
plt.subplot(3, 2, 5)
plt.scatter(comparison_log['normal_med'], comparison_log['normal_large'])
plt.title('Log Transformation')
plt.xlabel('log(normal_med)')
plt.ylabel('log(normal_large)')

# Normalized (L2)
plt.subplot(3, 2, 6)
plt.scatter(comparison_norm['normal_med'], comparison_norm['normal_large'])
plt.title('Normalization (L2)')
plt.xlabel('normal_med (normalized)')
plt.ylabel('normal_large (normalized)')

plt.tight_layout()
plt.show()

# Create a summary table of each method's characteristics
methods = ['Original', 'Min-Max Scaling', 'Standardization', 'Robust Scaling', 'Log Transform', 'Normalization (L2)']
dataframes = [comparison_df_outliers, comparison_minmax, comparison_std, comparison_robust, comparison_log, comparison_norm]

# Calculate various statistics
stats = []
for i, df in enumerate(dataframes):
    stats.append({
        'Method': methods[i],
        'Range (normal_med)': f"{df['normal_med'].min():.2f} to {df['normal_med'].max():.2f}",
        'Range (normal_large)': f"{df['normal_large'].min():.2f} to {df['normal_large'].max():.2f}",
        'Mean (normal_med)': f"{df['normal_med'].mean():.2f}",
        'Mean (normal_large)': f"{df['normal_large'].mean():.2f}",
        'Std (normal_med)': f"{df['normal_med'].std():.2f}",
        'Std (normal_large)': f"{df['normal_large'].std():.2f}",
        'Correlation': f"{df['normal_med'].corr(df['normal_large']):.2f}"
    })

stats_df = pd.DataFrame(stats)
print(stats_df)

## 10. Scaling in Machine Learning Pipelines

It's crucial to incorporate feature scaling correctly in machine learning workflows. One key consideration is to avoid data leakage by applying the scaling only to the training data and then using the same parameters to transform the test data.

Scikit-learn's `Pipeline` is an excellent tool for ensuring proper scaling in the ML workflow. Let's demonstrate how to use it.

In [ ]:
# Load a real dataset for our machine learning example
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target

print("Original feature statistics:")
print(X.describe().loc[['min', 'max', 'mean', 'std']].T)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# INCORRECT APPROACH: Scaling before splitting (data leakage)
# This is incorrect because information from test data leaks into the scaling parameters
X_scaled_incorrect = StandardScaler().fit_transform(X)
X_train_incorrect, X_test_incorrect, y_train_incorrect, y_test_incorrect = train_test_split(
    X_scaled_incorrect, y, test_size=0.3, random_state=42)

# CORRECT APPROACH 1: Scale after splitting
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit and transform on training data
X_test_scaled = scaler.transform(X_test)  # Only transform on test data

# CORRECT APPROACH 2: Using Pipeline
# This ensures that scaling is done properly during cross-validation
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Step 1: Scale the features
    ('model', SVC(kernel='rbf'))   # Step 2: Train an SVM model
])

# Compare different approaches
# 1. Train SVM without scaling
svm_no_scale = SVC(kernel='rbf')
scores_no_scale = cross_val_score(svm_no_scale, X, y, cv=5, scoring='neg_mean_squared_error')

# 2. Train SVM with incorrect scaling (data leakage)
svm_incorrect = SVC(kernel='rbf')
scores_incorrect = cross_val_score(svm_incorrect, X_scaled_incorrect, y, cv=5, scoring='neg_mean_squared_error')

# 3. Train SVM with correct scaling using pipeline
scores_pipeline = cross_val_score(pipeline, X, y, cv=5, scoring='neg_mean_squared_error')

# Compare the results
print("\nCross-validation MSE scores (negative, lower is better):")
print("Without scaling:", np.mean(scores_no_scale), "±", np.std(scores_no_scale))
print("With incorrect scaling (leakage):", np.mean(scores_incorrect), "±", np.std(scores_incorrect))
print("With correct scaling (pipeline):", np.mean(scores_pipeline), "±", np.std(scores_pipeline))

# Demonstrate the pipeline steps in action
# We can access the scaler after fitting
pipeline.fit(X_train, y_train)
scaler_from_pipeline = pipeline.named_steps['scaler']

# Print scaling parameters to show they're from training data only
print("\nScaling parameters from pipeline (fitted on training data only):")
print("Mean values:", scaler_from_pipeline.mean_)
print("Scale values:", scaler_from_pipeline.scale_)

# Compare with scaler fitted on all data (incorrect approach)
incorrect_scaler = StandardScaler().fit(X)
print("\nScaling parameters from incorrect approach (fitted on all data):")
print("Mean values:", incorrect_scaler.mean_)
print("Scale values:", incorrect_scaler.scale_)

# Show that the parameters are different
mean_diff = np.abs(scaler_from_pipeline.mean_ - incorrect_scaler.mean_).mean()
scale_diff = np.abs(scaler_from_pipeline.scale_ - incorrect_scaler.scale_).mean()
print("\nAverage difference in parameters:")
print("Mean difference:", mean_diff)
print("Scale difference:", scale_diff)

## 11. Handling Outliers

Outliers can significantly impact the performance of machine learning models and the effectiveness of scaling techniques. Let's explore different approaches to handle outliers during the scaling process.

Strategies include:
1. Using robust scaling methods (as seen earlier with RobustScaler)
2. Removing or capping outliers before scaling
3. Transforming features (e.g., log transformation)

In [ ]:
# Create a dataset with outliers for demonstration
np.random.seed(42)
n_samples = 1000
n_outliers = 20

# Create feature with standard normal distribution
X_clean = np.random.normal(0, 1, n_samples)

# Add outliers
outlier_indices = np.random.choice(n_samples, n_outliers, replace=False)
X_with_outliers = X_clean.copy()
X_with_outliers[outlier_indices] = np.random.uniform(10, 15, n_outliers)

# Different methods to handle outliers
# 1. No special treatment (use StandardScaler)
std_scaler = StandardScaler()
X_std_scaled = std_scaler.fit_transform(X_with_outliers.reshape(-1, 1)).flatten()

# 2. Use RobustScaler
robust_scaler = RobustScaler()
X_robust_scaled = robust_scaler.fit_transform(X_with_outliers.reshape(-1, 1)).flatten()

# 3. Remove outliers before scaling (using z-score)
z_scores = np.abs((X_with_outliers - np.mean(X_with_outliers)) / np.std(X_with_outliers))
X_removed_outliers = X_with_outliers[z_scores < 3]  # Keep only non-outliers
std_scaler_no_outliers = StandardScaler()
X_std_no_outliers = std_scaler_no_outliers.fit_transform(X_removed_outliers.reshape(-1, 1)).flatten()

# 4. Cap outliers (winsorization)
lower_bound = np.percentile(X_with_outliers, 1)
upper_bound = np.percentile(X_with_outliers, 99)
X_capped = np.clip(X_with_outliers, lower_bound, upper_bound)
std_scaler_capped = StandardScaler()
X_std_capped = std_scaler_capped.fit_transform(X_capped.reshape(-1, 1)).flatten()

# 5. Log transformation
# Add a constant to ensure all values are positive
min_val = np.min(X_with_outliers)
X_shifted = X_with_outliers - min_val + 1 if min_val <= 0 else X_with_outliers
X_log = np.log(X_shifted)
std_scaler_log = StandardScaler()
X_log_scaled = std_scaler_log.fit_transform(X_log.reshape(-1, 1)).flatten()

# Visualize the results
plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.hist(X_with_outliers, bins=30)
plt.title('Original Data with Outliers')
plt.xlim(-5, 16)

plt.subplot(2, 3, 2)
plt.hist(X_std_scaled, bins=30)
plt.title('Standard Scaling')

plt.subplot(2, 3, 3)
plt.hist(X_robust_scaled, bins=30)
plt.title('Robust Scaling')

plt.subplot(2, 3, 4)
plt.hist(X_std_no_outliers, bins=30)
plt.title('Standard Scaling (Outliers Removed)')

plt.subplot(2, 3, 5)
plt.hist(X_std_capped, bins=30)
plt.title('Standard Scaling (Outliers Capped)')

plt.subplot(2, 3, 6)
plt.hist(X_log_scaled, bins=30)
plt.title('Log Transform + Standard Scaling')

plt.tight_layout()
plt.show()

# Box plots to better visualize the outliers in each approach
plt.figure(figsize=(15, 6))

methods = ['Original', 'Standard Scaled', 'Robust Scaled', 
           'Outliers Removed', 'Outliers Capped', 'Log + Standard']
data = [X_with_outliers, X_std_scaled, X_robust_scaled, 
        X_std_no_outliers, X_std_capped, X_log_scaled]

plt.boxplot(data, labels=methods)
plt.title('Comparison of Outlier Handling Methods')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Analyze statistical properties
stats = pd.DataFrame({
    'Method': methods,
    'Mean': [np.mean(d) for d in data],
    'Median': [np.median(d) for d in data],
    'Std Dev': [np.std(d) for d in data],
    'Min': [np.min(d) for d in data],
    'Max': [np.max(d) for d in data],
    'IQR': [np.percentile(d, 75) - np.percentile(d, 25) for d in data],
    'Skewness': [pd.Series(d).skew() for d in data]
})

print("Statistical properties of different scaling methods:")
print(stats)

## 12. When to Use Each Scaling Method

Choosing the right scaling method depends on your data characteristics, the algorithms you're using, and your specific use case. Here's a guide to help you decide when to use each scaling method.

In [ ]:
# Create a DataFrame to summarize each scaling method, its characteristics, advantages,
# limitations, and when to use it

scaling_methods = pd.DataFrame({
    'Method': [
        'Min-Max Scaling (Normalization)',
        'Standardization (Z-score)',
        'Robust Scaling',
        'Unit Vector Normalization',
        'Log Transformation'
    ],
    'Output Range': [
        'Fixed range, typically [0,1] or [-1,1]',
        'No fixed range, μ=0 and σ=1',
        'No fixed range, centered around median',
        'Each sample has length = 1',
        'Depends on the range of original data'
    ],
    'Preserves Distribution': [
        'No',
        'No, but preserves shape for normal distributions',
        'Partially',
        'No',
        'No, but makes skewed distributions more symmetric'
    ],
    'Sensitivity to Outliers': [
        'High',
        'Medium',
        'Low',
        'Medium',
        'Low (reduces the effect of outliers)'
    ],
    'Best Used With': [
        'Neural networks, algorithms requiring bounded inputs, image processing',
        'PCA, linear regression, logistic regression, neural networks',
        'Datasets with outliers, non-Gaussian distributions',
        'Text data, cosine similarity, SVM, algorithms using dot products',
        'Right-skewed data, data with exponential growth, power-law distributions'
    ],
    'Avoid When': [
        'Data has outliers, non-linear transformations needed',
        'Data is not normally distributed, has significant outliers',
        'Data is normally distributed without outliers',
        'Scale differences between features matter',
        'Data contains zeros or negative values (without adjustment)'
    ]
})

print("Scaling Method Selection Guide:")
print(scaling_methods)

# Create a decision flowchart as a visualization
# This is a simplified representation of a flowchart as text
flowchart_text = """
Decision Flowchart for Choosing a Scaling Method:

Is your algorithm distance-based (K-means, KNN) or does it use gradient descent?
├── Yes → Does your data have outliers?
│       ├── Yes → Use Robust Scaling
│       └── No → Is your data normally distributed?
│              ├── Yes → Use Standardization
│              └── No → Use Min-Max Scaling
└── No → Is it text data or are you using cosine similarity?
        ├── Yes → Use Unit Vector Normalization
        └── No → Is your data highly skewed?
               ├── Yes → Use Log Transformation (ensure data is positive)
               └── No → Is the algorithm sensitive to feature magnitudes?
                      ├── Yes → Use Standardization or Min-Max Scaling
                      └── No → No scaling needed
"""

print("\nDecision Guide:")
print(flowchart_text)

# Create a table of algorithms and their recommended scaling methods
ml_algorithms = pd.DataFrame({
    'Algorithm': [
        'Linear Regression',
        'Logistic Regression',
        'Support Vector Machine',
        'K-Nearest Neighbors',
        'K-Means Clustering',
        'Decision Trees',
        'Random Forest',
        'Gradient Boosting',
        'Neural Networks',
        'PCA/LDA',
        'Naive Bayes'
    ],
    'Scaling Required?': [
        'Yes',
        'Yes',
        'Yes',
        'Yes',
        'Yes',
        'No',
        'No',
        'No',
        'Yes',
        'Yes',
        'Depends on implementation'
    ],
    'Recommended Method': [
        'Standardization',
        'Standardization',
        'Standardization or Min-Max',
        'Min-Max or Standardization',
        'Standardization',
        'None (not sensitive to scaling)',
        'None (not sensitive to scaling)',
        'None (not sensitive to scaling)',
        'Min-Max or Standardization',
        'Standardization',
        'Features should be on same scale if using Gaussian NB'
    ]
})

print("\nScaling Recommendations for Different ML Algorithms:")
print(ml_algorithms)

# Create a simple example to demonstrate algorithm performance with different scaling methods
def create_example_dataset():
    # Create a synthetic dataset with different scales
    X, y = make_classification(
        n_samples=1000, 
        n_features=5, 
        n_informative=3,
        n_redundant=2,
        random_state=42
    )
    
    # Scale features differently
    X[:, 0] = X[:, 0] * 0.1  # Small scale
    X[:, 1] = X[:, 1] * 100   # Large scale
    X[:, 2] = X[:, 2] * 0.01  # Very small scale
    X[:, 3] = X[:, 3] * 50    # Medium-large scale
    
    return X, y

# Generate the dataset
X, y = create_example_dataset()

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create different pipelines with different scaling methods
pipelines = {
    'No Scaling': Pipeline([
        ('model', SVC(kernel='rbf', gamma='auto'))
    ]),
    'Min-Max Scaling': Pipeline([
        ('scaler', MinMaxScaler()),
        ('model', SVC(kernel='rbf', gamma='auto'))
    ]),
    'Standardization': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', gamma='auto'))
    ]),
    'Robust Scaling': Pipeline([
        ('scaler', RobustScaler()),
        ('model', SVC(kernel='rbf', gamma='auto'))
    ])
}

# Train and evaluate each pipeline
results = {}
for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    train_score = pipeline.score(X_train, y_train)
    test_score = pipeline.score(X_test, y_test)
    results[name] = (train_score, test_score)

# Visualize the results
methods = list(results.keys())
train_scores = [results[method][0] for method in methods]
test_scores = [results[method][1] for method in methods]

plt.figure(figsize=(10, 6))
bar_width = 0.35
x = np.arange(len(methods))

plt.bar(x - bar_width/2, train_scores, bar_width, label='Training Accuracy')
plt.bar(x + bar_width/2, test_scores, bar_width, label='Testing Accuracy')

plt.xlabel('Scaling Method')
plt.ylabel('Accuracy')
plt.title('SVM Performance with Different Scaling Methods')
plt.xticks(x, methods)
plt.legend()

plt.tight_layout()
plt.show()

print("SVM Performance Results:")
for method in methods:
    train_acc, test_acc = results[method]
    print(f"{method}: Train Accuracy = {train_acc:.4f}, Test Accuracy = {test_acc:.4f}")

## Summary

In this notebook, we've explored various feature scaling and normalization techniques used in machine learning:

1. **Min-Max Scaling**: Rescales features to a fixed range (usually [0, 1])
   - Best for neural networks and algorithms requiring bounded inputs
   - Sensitive to outliers

2. **Standardization (Z-score)**: Rescales to zero mean and unit variance
   - Ideal for algorithms assuming normally distributed data
   - Good for PCA, linear models, and when features have different scales

3. **Robust Scaling**: Uses median and IQR instead of mean and std
   - Excellent for data with outliers
   - More robust than standardization for skewed distributions

4. **Unit Vector Normalization**: Scales samples to unit norm
   - Useful for text data and cosine similarity
   - Good for SVM and algorithms using dot products

5. **Log Transformation**: Handles skewed data and reduces effect of outliers
   - Great for right-skewed data and features with exponential growth
   - Makes multiplicative relationships more additive

**Key Takeaways:**
- Always scale your features for distance-based and gradient-based algorithms
- Apply scaling only to training data, then use the same parameters for test data
- Consider the nature of your data and algorithm requirements when choosing a scaling method
- Be aware of how outliers affect different scaling techniques
- Use scikit-learn's Pipeline to ensure proper implementation of scaling in machine learning workflows

Feature scaling is a critical preprocessing step that can significantly improve model performance, convergence speed, and stability. Choosing the right scaling method for your specific data and algorithm can make a substantial difference in your machine learning projects.